<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w5_reranking/llm_260409_reranking_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 260409 A/B Test + LLM-as-Judge

**5주차 3일차** | 리랭킹 효과를 통계적으로 검증하고, LLM을 평가자로 활용하기

---

## 오늘 배울 내용

| 주제 | 핵심 | 비유 |
|------|------|------|
| **A/B Test (t-test)** | 리랭킹 전/후 성능 차이가 "진짜"인지 통계 검증 | 약 효과 실험: 위약 vs 진짜 약, 우연인지 진짜인지 구분 |
| **Cohen's d** | 차이가 있다면 "얼마나" 큰지 | 키 차이: 1cm 차이 vs 10cm 차이 |
| **LLM-as-Judge** | LLM에게 답변 품질 평가를 시키기 | AI 채점관: BLEU/ROUGE 대신 의미를 이해하는 평가 |
| **Pointwise** | 답변 1개씩 절대 점수 매기기 | 시험 채점: 각 답안지를 독립적으로 채점 |
| **Pairwise** | 답변 2개를 비교해서 승자 결정 | 토너먼트: A vs B 누가 더 나은가 |
| **Reference-based** | 정답 기준으로 답변 평가 | 모범답안 대비 학생 답안 채점 |

---

### 흐름 요약
```
w5d1: 리랭킹 방법론 (BM25, LLM, Hybrid, Listwise)
w5d2: Cross-encoder + 평가 메트릭 (MAP, ILS)
w5d3: A/B Test로 통계 검증 + LLM-as-Judge 평가 <-- 오늘!
```

## 0. Setup

In [1]:
!pip install -q faiss-cpu langchain langchain-community langchain-core langchain-openai matplotlib numpy openai pandas rank-bm25 scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from dotenv import load_dotenv

# load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 문서 및 벡터스토어 준비

w5d1~d2에서 사용한 동일한 문서셋 + FAISS 벡터스토어 + BM25 리트리버

In [4]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성으로 외부 문서를 활용하여 LLM 답변을 강화합니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 검색은 임베딩 공간에서 코사인 유사도로 문서를 찾는 방법입니다.", metadata={"id": "d5"}),
    Document(page_content="LLM 학습에는 사전학습과 미세조정 두 단계가 있으며 대규모 데이터가 필요합니다.", metadata={"id": "d6"}),
    Document(page_content="RLHF는 인간 피드백을 활용한 강화학습으로 LLM의 응답 품질을 향상시킵니다.", metadata={"id": "d7"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d8"}),
]

vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [5]:
# 문서 임베딩 미리 계산 (ILS 등에 필요)
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in documents:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

## 2. 이전 수업 함수 복습 (벡터 검색 + BM25 리랭커 + AP/MAP)

w5d1~d2에서 구현한 함수들을 재사용합니다.

In [6]:
# --- 벡터 검색 ---
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1/(1+score)) for doc, score in results]

# --- BM25 리랭커 (w5d1) ---
class BM25Reranker:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def rerank(self, query, search_results):
        query_terms = query.lower().split()
        doc_lengths = [len(doc.page_content.split()) for doc, _ in search_results]
        avg_dl = np.mean(doc_lengths) if doc_lengths else 1

        scored = []
        for doc, orig_score in search_results:
            words = doc.page_content.lower().split()
            dl = len(words)
            score = 0
            for term in query_terms:
                tf = words.count(term)
                idf = 1.0
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * dl / avg_dl)
                score += idf * numerator / denominator
            scored.append((doc, orig_score + score))
        return sorted(scored, key=lambda x: x[1], reverse=True)

# --- Average Precision ---
def average_precision(retrieved_ids, relevant_ids):
    """검색 결과의 AP (Average Precision) 계산"""
    relevant_set = set(relevant_ids)
    hits = 0
    sum_precision = 0.0
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_set:
            hits += 1
            precision_at_i = hits / (i + 1)
            sum_precision += precision_at_i
    return sum_precision / len(relevant_set) if relevant_set else 0.0

# --- Mean Average Precision ---
def mean_average_precision(query_results):
    """여러 쿼리의 MAP 계산"""
    aps = []
    for query, (retrieved, relevant) in query_results.items():
        ap = average_precision(retrieved, relevant)
        aps.append(ap)
    return np.mean(aps)

---

## 3. A/B Test: 리랭킹 효과를 통계적으로 검증하기

### 핵심 개념

리랭킹을 추가했더니 MAP이 0.82 -> 0.69로 변했다. 이게 **진짜 차이**일까, 아니면 **우연**일까?

> **비유**: 새 약을 먹었더니 두통이 나았다. 약 효과일까, 그냥 시간이 지나서 나은 걸까?
> 이걸 판별하는 게 **t-test**!

### t-test 핵심 정리

| 용어 | 의미 | 비유 |
|------|------|------|
| **귀무가설 (H0)** | A와 B에 차이가 없다 | "약은 효과 없다" (기본 가정) |
| **대립가설 (H1)** | A와 B에 차이가 있다 | "약은 효과 있다" |
| **p-value** | H0이 참인데 이 결과가 나올 확률 | 0.01이면 = 차이 없는데 이렇게 나올 확률 1% |
| **유의수준 (alpha)** | 보통 0.05 (5%) | p-value < alpha면 "차이 있다!" 판정 |
| **Cohen's d** | 차이의 크기 | 0.2=작다, 0.5=보통, 0.8=크다 |

```
p-value가 작다 = 우연이 아니다 = 귀무가설 기각 = 진짜 차이가 있다!
```

### 주의: 샘플 수가 적으면 p-value가 높게 나온다
- 수업에서 쿼리 3개로 테스트 -> p-value가 0.6 (차이 없음 판정)
- **50개 이상** 쿼리로 해야 신뢰할 수 있는 결과

In [7]:
# --- A/B Test 함수 ---
# scipy.stats.ttest_rel: 대응 표본 t-test (같은 쿼리셋에 대해 A방법 vs B방법 비교)
#
# Cohen's d = (평균A - 평균B) / 공통 표준편차
#   -> 두 집단의 "거리"를 표준편차로 나눠서 정규화
#   -> 비유: 키 차이 5cm가 크다? 작다? -> 사람들 키의 편차(~7cm) 대비로 판단

def ab_test(scores_a, scores_b, alpha=0.05):
    """A/B 테스트: t-test + Cohen's d"""
    # 대응 표본 t-test (같은 쿼리에 대한 전/후 비교이므로 ttest_rel 사용)
    t_stat, p_val = stats.ttest_rel(scores_a, scores_b)

    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)

    # 공통 표준편차 (pooled std): 두 그룹이 얼마나 퍼져있는지
    pooled_std = np.sqrt((np.std(scores_a)**2 + np.std(scores_b)**2) / 2)
    cohens_d = (mean_a - mean_b) / pooled_std if pooled_std > 0 else 0.0

    return {
        'mean_A': mean_a,
        'mean_B': mean_b,
        'p_val': p_val,
        'cohens_d': cohens_d,
        'significant': p_val < alpha   # True면 "차이 있다!"
    }

In [8]:
# --- 실험: 벡터 검색 vs BM25 리랭킹 ---
test_queries = {
    '트랜스포머 아키텍처': ({'d1', 'd2', 'd3'}),
    '벡터 검색 방법': ({'d5'}),
    'LLM 학습 방법': ({'d6', 'd7'})
}

aps_before = []  # 벡터 검색만
aps_after = []   # BM25 리랭킹 후
bm25_reranker = BM25Reranker()

for q, relevant in test_queries.items():
    orig = vector_search(q, vectorstore)
    reranked = bm25_reranker.rerank(q, orig)

    orig_ids = [doc.metadata['id'] for doc, _ in orig]
    reranked_ids = [doc.metadata['id'] for doc, _ in reranked]

    aps_before.append(average_precision(orig_ids, relevant))
    aps_after.append(average_precision(reranked_ids, relevant))

print(f"Before (vector only) AP scores: {aps_before}")
print(f"After  (BM25 rerank) AP scores: {aps_after}")

Before (vector only) AP scores: [1.0, 1.0, 1.0]
After  (BM25 rerank) AP scores: [1.0, 1.0, 0.8333333333333333]


In [9]:
# A/B 테스트 실행
result = ab_test(aps_before, aps_after, alpha=0.05)
result

{'mean_A': np.float64(1.0),
 'mean_B': np.float64(0.9444444444444443),
 'p_val': np.float64(0.42264973081037427),
 'cohens_d': np.float64(1.000000000000002),
 'significant': np.False_}

### 결과 해석

- `p_val`이 0.05보다 **크면** -> `significant: False` -> 차이가 있다고 말하기 어려움
- 쿼리가 3개뿐이라 p-value가 높게 나옴 (샘플 부족)
- 실무에서는 **50개 이상의 쿼리**로 테스트해야 신뢰할 수 있음

> **비유**: 동전을 3번 던져서 앞면 2번 나왔다고 "이 동전은 앞면이 잘 나온다"라고 말할 수 없음.
> 100번 던져서 70번 나오면 그때 "이상한 동전"이라고 할 수 있음.

---

## 4. LLM-as-Judge: LLM을 평가자로 활용하기

### 왜 LLM-as-Judge인가?

기존 메트릭(BLEU, ROUGE)의 한계:
- **n-gram 기반**: 단어가 겹쳐야만 점수가 오름
- "딥러닝은 심층 신경망" vs "deep learning은 깊은 neural network" -> ROUGE 0점이지만 같은 의미!
- 원래 번역 평가용으로 만들어진 것

LLM-as-Judge의 장점:
- **의미 이해** 가능 (semantic understanding)
- **유연한 평가** 기준 설정
- 복잡한 품질도 평가 가능

LLM-as-Judge의 단점:
- **비용** (API 호출)
- **기준이 모호** (블랙박스)
- **재현성** 부족 (같은 입력에 다른 점수)

### 세 가지 방법

```
1. Pointwise : 하나씩 절대 점수 (시험 채점)
2. Pairwise  : 둘을 비교해서 승자 (토너먼트)
3. Reference-based : 정답 기준 평가 (모범답안 채점)
```

### 4-1. Basic Judge: 가장 간단한 LLM 평가

In [10]:
def basic_judge(question, answer, scale='1~5'):
    """기본 LLM 평가: 질문-답변 쌍을 점수화"""
    prompt = f"""다음 답변을 {scale}스케일로 평가하세요.

    질문 : {question}
    답변 : {answer}

    평가 기준
    - 정확성 : 사실적으로 올바른가?
    - 완전성 : 질문에 충분히 답했는가?
    - 명확성 : 이해하기 쉽게 설명했는가?

    반드시 아래 형식으로 답하세요:
    점수 : [숫자]
    이유 : [한 줄 설명]"""

    result = llm.invoke(prompt).content
    return result

In [11]:
# 테스트: 좋은 답변 / 짧은 답변 / 엉뚱한 답변
question = "파이썬의 장점은 무엇인가요?"
answers = [
    "파이썬은 간결한 문법으로 초보자도 쉽게 배울 수 있으며, 풍부한 라이브러리와 활발한 커뮤니티가 있습니다.",
    "파이썬 좋아요.",
    "자바스크립트는 웹 개발에 주로 사용됩니다.",
]

for i, ans in enumerate(answers):
    result = basic_judge(question, ans)
    print(f"답변 {i+1}: {ans[:30]}...")
    print(f"평가: {result}")
    print()

답변 1: 파이썬은 간결한 문법으로 초보자도 쉽게 배울 수 있으며...
평가: 점수 : 5  
이유 : 파이썬의 장점에 대해 정확하고 완전하며 명확하게 설명하였습니다.

답변 2: 파이썬 좋아요....
평가: 점수 : 1  
이유 : 답변이 매우 간단하고 파이썬의 장점에 대한 구체적인 정보가 전혀 제공되지 않아 질문에 제대로 답하지 못함.

답변 3: 자바스크립트는 웹 개발에 주로 사용됩니다....
평가: 점수 : 1  
이유 : 답변이 질문과 전혀 관련이 없고, 파이썬의 장점에 대한 정보가 전혀 포함되어 있지 않기 때문입니다.



### 4-2. ROUGE vs LLM-as-Judge 비교

ROUGE는 n-gram(단어) 겹침만 보기 때문에 한계가 있음:
- 정답 복사 -> ROUGE 1.0, LLM 5점 (당연)
- 좋은 패러프레이즈 -> ROUGE **0.0** (!), LLM **4점** (의미는 이해)
- 키워드 나열 -> ROUGE 0.1, LLM **2점** (문장이 아님)

> ROUGE가 못 잡는 걸 LLM이 잡아준다!

In [12]:
# --- 간단한 ROUGE-1 F1 (유니그램 기반) ---
def simple_rouge_f1(reference, candidate):
    """ROUGE-1 F1: 단어 단위로 겹치는 비율"""
    ref_words = Counter(reference.split())
    cand_words = Counter(candidate.split())
    overlap = sum(min(ref_words[w], cand_words.get(w, 0)) for w in ref_words)
    r = overlap / sum(ref_words.values()) if ref_words else 0  # recall
    p = overlap / sum(cand_words.values()) if cand_words else 0  # precision
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

In [13]:
# --- LLM Judge (JSON 반환 버전) ---
def basic_judge2(question, answer, scale='1~5'):
    """LLM 평가: JSON으로 score + confidence 반환"""
    prompt = f"""다음 답변을 {scale}스케일로 평가하세요. 답변은 반드시 JSON으로 반환하세요

    질문 : {question}
    답변 : {answer}

    평가 기준
    - 정확성 : 사실적으로 올바른가?
    - 완전성 : 질문에 충분히 답했는가?
    - 명확성 : 이해하기 쉽게 설명했는가?

    반드시 아래 형식으로 답하세요:
    {{"score": 1~5점수, "confidence": "high/mid/low"}}"""

    result = llm.invoke(prompt).content
    cleaned = result.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
    return json.loads(cleaned)

In [14]:
# --- ROUGE vs LLM 비교 실험 ---
question = "딥러닝이란?"
reference = "딥러닝은 다층 신경망을 사용하여 데이터에서 복잡한 패턴을 학습하는 머신러닝 방법입니다"

test_answers = [
    ("딥러닝은 다층 신경망을 사용하여 데이터에서 복잡한 패턴을 학습하는 머신러닝 방법입니다", "정답 복사"),
    ("인공 신경망의 층을 깊게 쌓아 복잡한 문제를 풀 수 있는 AI 기술이에요", "좋은 패러프레이즈"),
    ("딥러닝 딥러닝 신경망 데이터 학습 패턴 머신러닝", "키워드 나열"),
]

rows = []
for ans, label in test_answers:
    rouge = simple_rouge_f1(reference, ans)
    llm_result = basic_judge2(question, ans)
    print(f"[{label}] ROUGE={rouge:.2f}, LLM score={llm_result.get('score', 0)}")
    rows.append({
        '유형': label,
        'rouge': round(rouge, 3),
        'llm_score': llm_result.get('score', 0),
        'llm_confidence': llm_result.get('confidence', '?')
    })

pd.DataFrame(rows)

[정답 복사] ROUGE=1.00, LLM score=4
[좋은 패러프레이즈] ROUGE=0.09, LLM score=4
[키워드 나열] ROUGE=0.12, LLM score=2


,유형,rouge,llm_score,llm_confidence
0,정답 복사,1.000,4,high
1,좋은 패러프레이즈,0.091,4,mid
2,키워드 나열,0.118,2,low


---

## 5. Pointwise Judge: 루브릭 기반 절대 평가

> **비유**: 수능 채점 - 미리 정해진 채점 기준(루브릭)에 따라 각 답안을 독립적으로 채점

- 장점: 확장 쉬움 (N개 답변 = N번 호출)
- 단점: LLM의 절대 기준이 불안정 (같은 답변에 어떤 때는 4점, 어떤 때는 3점)

In [15]:
# --- 루브릭: 각 점수의 명확한 기준 ---
RUBRIC = {
    5: "정확하고 완전하며, 예시와 설명이 풍부하다",
    4: "정확하고 핵심을 다루지만, 일부 세부사항이 부족하다",
    3: "대체로 정확하지만, 중요한 내용 일부가 누락되었다",
    2: "부분적으로만 정확하거나, 핵심을 빗나갔다",
    1: "부정확하거나 질문과 무관하다",
}

def pointwise_judge(question, answer, rubric=RUBRIC):
    """루브릭 기반 Pointwise 평가"""
    rubric_text = '\n'.join(
        f'  {k}점: {v}' for k, v in sorted(rubric.items(), reverse=True)
    )

    prompt = f"""다음 답변을 아래 루브릭에 따라 평가해주세요.

    질문 : {question}
    답변 : {answer}

    루브릭 :
    {rubric_text}

    반드시 JSON으로 답하세요:
    {{"score": 1-5, "reasoning": "이유"}}"""

    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [16]:
# Pointwise 평가 테스트
question = "REST API와 GraphQL의 차이점을 설명해주세요"
answers = [
    "REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQL은 단일 엔드포인트에서 클라이언트가 필요한 데이터를 쿼리합니다. REST는 오버/언더 페칭 문제가 있고, GraphQL은 이를 해결하지만 캐싱이 어렵습니다.",
    "REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다.",
    "둘 다 API입니다.",
]

for i, ans in enumerate(answers):
    result = pointwise_judge(question, ans)
    print(f"답변 {i+1}: {ans[:40]}...")
    print(f"  점수: {result.get('score')}/5 - {result.get('reasoning', '')[:50]}")
    print()

답변 1: REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQ...
  점수: 4/5 - 답변은 REST와 GraphQL의 핵심 차이를 잘 설명하고 있지만, 예시가 부족하고 각 기

답변 2: REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다....
  점수: 2/5 - 답변은 부분적으로만 정확하지만, REST API와 GraphQL의 주요 차이점인 요청 방식

답변 3: 둘 다 API입니다....
  점수: 1/5 - 답변이 질문에 대한 내용과 핵심적인 차이를 전혀 다루지 않고, '둘 다 API입니다'라는 



---

## 6. Pairwise Judge: 상대 비교 평가

> **비유**: 토너먼트 - "A와 B 중 누가 더 나은가?" 절대 점수 없이 비교만!

- 장점: 비교가 절대 평가보다 쉽고 일관적
- 단점: O(N^2) 복잡도 (5개면 10번, 500개면 125,000번!)
- **위치 편향 문제**: LLM은 **앞에 나온 답변을 선호**하는 경향 (Lost-in-the-middle과 같은 원리)
  - 해결: A,B 순서를 바꿔서 2번 평가 -> 결과 일치하면 신뢰

In [17]:
# --- Pairwise Judge: 두 답변 비교 ---
def pairwise_judge(question, answer_a, answer_b):
    """두 답변을 비교하여 승자 결정"""
    prompt = f"""두 답변을 비교하여 더 나은쪽 선택하세요

    질문 : {question}

    답변 A : {answer_a}

    답변 B : {answer_b}

    반드시 JSON으로 답하세요:
    {{"winner": "A 또는 B 또는 Tie", "reasoning": "선택 이유"}}"""

    result = llm.invoke(prompt).content
    return json.loads(result.replace('```json', '').replace('```', ''))

In [18]:
# 테스트
question = "마이크로서비스 아키텍처란?"
answer_a = "마이크로서비스는 애플리케이션을 작은 독립 서비스로 분리하는 아키텍처 패턴입니다. 각 서비스는 독립 배포, 확장이 가능합니다."
answer_b = "여러 개의 작은 서비스로 나누는 것입니다."

# 정방향 비교
print("=== A먼저, B나중 ===")
r1 = pairwise_judge(question, answer_a, answer_b)
print(r1)

print("\n=== B먼저, A나중 ===")
r2 = pairwise_judge(question, answer_b, answer_a)
print(r2)

=== A먼저, B나중 ===
{'winner': 'A', 'reasoning': '답변 A는 마이크로서비스 아키텍처의 정의를 보다 구체적으로 설명하고 있으며, 독립 배포와 확장 가능성 등의 중요한 특성도 포함하고 있어 더 명확하고 유용한 정보 제공.'}

=== B먼저, A나중 ===
{'winner': 'B', 'reasoning': '답변 B는 마이크로서비스 아키텍처의 정의를 보다 구체적으로 설명하고 있으며, 각 서비스의 독립 배포 및 확장 가능성에 대해서도 언급하고 있습니다. 이는 사용자에게 아키텍처의 장점과 특징을 보다 명확히 전달합니다.'}


### 6-1. 위치 편향 보정 (Swap Test)

순서를 바꿔서 2번 평가한 뒤, 결과가 일치하는지 확인

In [19]:
def pairwise_judge_with_swap(question, answer_a, answer_b):
    """순서 바꿔서 2번 평가 -> 위치 편향 보정"""
    # 1차: A, B 순서
    result1 = pairwise_judge(question, answer_a, answer_b)
    # 2차: B, A 순서 (뒤집기)
    result2 = pairwise_judge(question, answer_b, answer_a)

    # 2차 결과의 winner를 원래 기준으로 변환
    swapped_winner = {'A': 'B', 'B': 'A', 'Tie': 'Tie'}.get(result2.get('winner', 'Tie'))

    if result1.get('winner') == swapped_winner:
        final = result1['winner']
        status = '일치'  # 위치 바꿔도 같은 결과 -> 신뢰 가능
    else:
        final = 'Tie'
        status = '불일치'  # 위치에 따라 달라짐 -> 편향 발생

    return {'final': final, 'status': status}

In [20]:
# 위치 편향 보정된 Pairwise 평가
pairwise_judge_with_swap(question, answer_a, answer_b)

{'final': 'A', 'status': '일치'}

### 6-2. 위치 편향 비율 측정

In [21]:
def detect_position_bias(question, answer_a, answer_b, n_trials=3):
    """위치 편향이 얼마나 자주 발생하는지 측정"""
    consistent = 0
    inconsistent = 0

    for trial in range(n_trials):
        result = pairwise_judge_with_swap(question, answer_a, answer_b)
        if result['status'] == '일치':
            consistent += 1
        else:
            inconsistent += 1

    total = consistent + inconsistent
    bias_rate = inconsistent / total
    return {
        'consistent': consistent,
        'inconsistent': inconsistent,
        'bias_rate': bias_rate
    }

In [22]:
# 위치 편향 측정 (n_trials번 반복)
detect_position_bias(question, answer_a, answer_b, n_trials=3)

{'consistent': 3, 'inconsistent': 0, 'bias_rate': 0.0}

---

## 7. Reference-based Judge: 정답 기준 평가

> **비유**: 모범답안이 있는 시험 채점 - 정답 대비 뭘 맞췄고, 뭘 빠뜨렸고, 뭘 틀렸는지

- 장점: 정답(레퍼런스)이 있으니 가장 객관적
- 단점: 좋은 정답을 만드는 비용이 큼

In [23]:
def reference_based_judge(question, reference, answer):
    """정답 기준 평가: 포함/누락/오류를 구분"""
    prompt = f"""당신은 AI 교육 전문가입니다.
    정답을 기준으로 학생의 답변을 평가하세요.

    질문 : {question}
    정답 : {reference}
    학생 답변 : {answer}

    평가 기준:
    - 정답의 핵심 포인트를 얼마나 포함하는가
    - 사실적으로 틀린 내용이 있는가
    - 정답에 없는 올바른 추가 정보가 있는가

    JSON으로 답하세요:
    {{"score": 1-5, "covered_points": ["포함된 핵심 포인트들"], "missing_points": ["누락된 포인트들"], "errors": ["틀린 내용"]}}"""

    result = llm.invoke(prompt).content
    cleaned = result.replace('```json', '').replace('```', '')
    return json.loads(cleaned)

In [24]:
# Reference-based 평가 테스트
question = "TCP와 UDP의 차이점은?"
reference = "TCP는 연결지향적이고 신뢰성 있는 전송을 보장하며, UDP는 비연결지향적이고 빠르지만 신뢰성을 보장하지 않습니다."
answer = "TCP는 연결을 먼저 수립하고 데이터 전송의 순서와 무결성을 보장합니다. 반면 UDP는 연결 없이 바로 전송하여 빠르지만 패킷 손실이 가능합니다."

result = reference_based_judge(question, reference, answer)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "score": 5,
  "covered_points": [
    "TCP는 연결을 먼저 수립한다",
    "TCP는 데이터 전송의 순서와 무결성을 보장한다",
    "UDP는 연결 없이 바로 전송한다",
    "UDP는 빠르지만 패킷 손실이 가능하다"
  ],
  "missing_points": [
    "TCP는 신뢰성 있는 전송을 보장한다",
    "UDP는 신뢰성을 보장하지 않는다"
  ],
  "errors": []
}


---

## 8. 정리

### 오늘 배운 것

| 도구 | 언제 쓰나 | 핵심 |
|------|-----------|------|
| **A/B Test (t-test)** | 리랭커/리트리버 변경 후 효과 검증 | p-value < 0.05면 유의미한 차이 |
| **Cohen's d** | 차이의 크기 측정 | 0.2=작다, 0.5=보통, 0.8=크다 |
| **Pointwise Judge** | 답변 품질을 개별 채점 | 루브릭 + 스케일, O(N) |
| **Pairwise Judge** | 두 시스템 직접 비교 | 위치 편향 보정 필요, O(N^2) |
| **Reference-based** | 정답이 있을 때 정밀 평가 | 포함/누락/오류 구분 |

### 실무 팁 (수업 강조 포인트)

1. **리트리버/리랭커 변경 시 반드시 정량 평가** -> MAP 같은 메트릭 + A/B Test
2. **비용/시간도 측정** -> 리랭킹은 비용이 높으니 효과 대비 가치 확인
3. **LLM-as-Judge는 편하지만 맹신 금지** -> 기준 모호 + 재현성 문제
4. **다음 수업 예고**: Graph RAG -- Naive RAG의 한계(멀티홉 질문 등)를 극복하는 방법

### 다음에 이어서
- w5d4: 추가 평가 실습 + 고급 RAG 전략
- w6: Graph RAG 도입